# 🏥 AI Medical Chatbot — Complete End-to-End Pipeline
### MedGemma 4B + QLoRA + RAG (FAISS + BGE) + Multi-Layer Safety + Gradio UI

> **⚠️ Medical Disclaimer:** This AI system is developed for educational and research purposes only. It is **not** a substitute for professional medical advice, diagnosis, or clinical treatment. For emergencies, always contact local emergency services immediately.

This notebook is an **all-in-one standalone Google Colab edition** containing the complete code for:
1. **Environment Setup & GPU Check**
2. **Medical Knowledge Base Documents & Sentence Chunking**
3. **FAISS Vector Indexing & Dense Semantic Retrieval (BGE Embeddings)**
4. **Cross-Encoder Reranking**
5. **Multi-Layer Clinical Safety & Emergency Detection Guardrails**
6. **MedGemma 4B QLoRA Fine-Tuning Setup**
7. **Accuracy, Classification Report & Confusion Matrix Evaluation**
8. **Interactive Gradio Web Chatbot UI (with public share link)**


---
## 1. Setup & Dependencies
Install all required libraries. Make sure to run this on a GPU runtime (**Runtime > Change runtime type > T4 GPU**).


In [ ]:
# Install all required dependencies
!pip install -q --upgrade \
    torch torchvision torchaudio \
    transformers accelerate peft trl bitsandbytes datasets \
    sentence-transformers faiss-cpu \
    gradio scikit-learn matplotlib seaborn pyyaml python-dotenv rich

import os
import re
import json
import time
import torch
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from enum import Enum, auto
from typing import List, Dict, Any, Optional

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device name     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM total      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


---
## 2. Medical Knowledge Base & Clinical Documents
We define foundational medical documents covering emergencies, chronic conditions, acute infections, and drug safety.


In [ ]:
MEDICAL_DOCUMENTS = [
    {
        "title": "Chest Pain: When Is It an Emergency?",
        "source": "American Heart Association",
        "category": "emergency",
        "text": (
            "CALL EMERGENCY SERVICES IMMEDIATELY (911) for: "
            "Chest pain with pressure, squeezing, tightness, or fullness in the center of the chest. "
            "Pain radiating to the left arm, jaw, neck, back, or stomach. "
            "Chest pain accompanied by shortness of breath, cold sweats, unexplained nausea, or lightheadedness. "
            "Sudden severe crushing chest pain at rest. "
            "These are hallmark symptoms of acute myocardial infarction (heart attack) or pulmonary embolism. "
            "Do NOT drive yourself to the emergency department; dispatch an ambulance immediately. "
            "Less urgent chest discomfort may stem from gastroesophageal reflux (heartburn), costochondritis, "
            "or anxiety/panic disorders, but new chest pain must always be medically triaged."
        )
    },
    {
        "title": "Hypertension (High Blood Pressure): Clinical Overview",
        "source": "AHA / CDC",
        "category": "cardiovascular",
        "text": (
            "High blood pressure (hypertension) is defined as consistent systolic pressure >= 130 mmHg "
            "or diastolic pressure >= 80 mmHg. It is frequently called the 'silent killer' because it "
            "rarely causes noticeable symptoms in early stages. "
            "Hypertensive crisis: A blood pressure reading above 180/120 mmHg is a medical emergency. "
            "If accompanied by severe headache, vision changes, chest pain, or shortness of breath, call 911. "
            "Long-term management involves sodium restriction (<2,300 mg/day), regular aerobic exercise, "
            "weight management, and prescribed antihypertensive medications (e.g., ACE inhibitors, ARBs, calcium channel blockers)."
        )
    },
    {
        "title": "Type 2 Diabetes Mellitus: Symptoms and Management",
        "source": "American Diabetes Association",
        "category": "metabolic",
        "text": (
            "Type 2 diabetes is a chronic metabolic disorder characterized by insulin resistance and relative insulin deficiency. "
            "Key clinical symptoms include polydipsia (excessive thirst), polyuria (frequent urination), polyphagia (constant hunger), "
            "unexplained weight loss, chronic fatigue, blurred vision, and slow-healing sores. "
            "Diagnostic thresholds: Fasting plasma glucose >= 126 mg/dL, or HbA1c >= 6.5%. "
            "Hypoglycemia (blood sugar < 70 mg/dL) causes shaking, sweating, confusion, and rapid heartbeat; "
            "it requires immediate treatment with 15 grams of fast-acting carbohydrates (Rule of 15)."
        )
    },
    {
        "title": "Fever Management & Warning Signs",
        "source": "Mayo Clinic / CDC",
        "category": "infectious_disease",
        "text": (
            "A fever is a temporary elevation in body temperature, usually in response to an infection. "
            "Normal body temperature is approximately 37°C (98.6°F). A clinical fever is defined as a core temperature >= 38.0°C (100.4°F). "
            "Mild fever treatment: adequate oral hydration, physical rest, and over-the-counter antipyretics "
            "(paracetamol/acetaminophen or ibuprofen) to alleviate discomfort. "
            "Red flags requiring urgent clinical care: fever > 39.4°C (103°F) in adults, fever lasting longer than 3 days, "
            "stiff neck with severe headache and confusion (meningitis signs), or any fever in infants under 3 months old."
        )
    },
    {
        "title": "Stroke: Signs, Symptoms, and F.A.S.T. Protocol",
        "source": "American Stroke Association",
        "category": "emergency",
        "text": (
            "A stroke occurs when blood flow to a region of the brain is interrupted (ischemic) or when a blood vessel ruptures (hemorrhagic). "
            "Use the F.A.S.T. acronym to detect stroke signs: "
            "F - Face Drooping (one side of the face droops or is numb). "
            "A - Arm Weakness (one arm drifts downward when both are raised). "
            "S - Speech Difficulty (slurred speech, unable to repeat simple sentences). "
            "T - Time to call 911 immediately. "
            "Do not wait to see if symptoms improve; emergency thrombolysis must be administered within hours of symptom onset."
        )
    },
    {
        "title": "Ibuprofen (NSAID): Clinical Use and Safety Warnings",
        "source": "FDA / NIH",
        "category": "medications",
        "text": (
            "Ibuprofen is a nonsteroidal anti-inflammatory drug (NSAID) used to treat pain, fever, and inflammation. "
            "Common side effects: dyspepsia, nausea, stomach pain, and gastrointestinal irritation. "
            "Serious risks: prolonged or high-dose NSAID use increases the risk of peptic ulcers, GI bleeding, and renal toxicity. "
            "Contraindications: patients with active stomach ulcers, advanced kidney disease, severe heart failure, "
            "or those in late pregnancy should avoid ibuprofen unless guided by a physician. "
            "Always take with food or milk to minimize gastric upset."
        )
    },
    {
        "title": "Iron Deficiency Anemia: Diagnosis and Treatment",
        "source": "WHO / MedlinePlus",
        "category": "hematology",
        "text": (
            "Iron deficiency anemia occurs when the body lacks sufficient iron to produce adequate hemoglobin for red blood cells. "
            "Typical clinical presentations: chronic fatigue, pale skin, weakness, shortness of breath on exertion, cold hands and feet, "
            "brittle nails, and pica (unusual cravings for non-nutritive substances like ice). "
            "Diagnosis is confirmed via complete blood count (CBC), serum ferritin, and transferrin saturation. "
            "Treatment includes dietary iron rich foods and oral iron supplementation (ferrous sulfate). Take with vitamin C for enhanced absorption."
        )
    }
]

print(f"Loaded {len(MEDICAL_DOCUMENTS)} foundational medical knowledge documents.")


---
## 3. Sentence-Aware Chunking & FAISS Vector Index
We chunk documents into dense 400-character segments (preserving full sentence boundaries) and embed them using `BAAI/bge-small-en-v1.5`.


In [ ]:
class MedicalChunker:
    """Split medical documents into sentence-preserving semantic chunks."""
    def __init__(self, chunk_size: int = 400, overlap: int = 50, min_chunk: int = 80):
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.min_chunk = min_chunk

    def split_sentences(self, text: str) -> List[str]:
        text = re.sub(r"\s+", " ", text).strip()
        protected = re.sub(r"\b(Dr|Mr|Mrs|Ms|vs|e\.g|i\.e)\.", r"\1<DOT>", text)
        sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z])", protected)
        return [s.replace("<DOT>", ".") for s in sentences if s.strip()]

    def chunk_document(self, doc: Dict[str, Any]) -> List[Dict[str, Any]]:
        sentences = self.split_sentences(doc["text"])
        chunks = []
        current_sentences = []
        current_chars = 0

        for sent in sentences:
            if current_chars + len(sent) > self.chunk_size and current_sentences:
                text_chunk = " ".join(current_sentences).strip()
                if len(text_chunk) >= self.min_chunk:
                    chunks.append({
                        "title": doc["title"],
                        "source": doc["source"],
                        "category": doc["category"],
                        "text": text_chunk
                    })
                current_sentences = [current_sentences[-1]] if len(current_sentences[-1]) <= self.overlap else []
                current_chars = sum(len(s) for s in current_sentences)

            current_sentences.append(sent)
            current_chars += len(sent)

        if current_sentences:
            text_chunk = " ".join(current_sentences).strip()
            if len(text_chunk) >= self.min_chunk:
                chunks.append({
                    "title": doc["title"],
                    "source": doc["source"],
                    "category": doc["category"],
                    "text": text_chunk
                })
        return chunks

chunker = MedicalChunker(chunk_size=400, overlap=50)
all_chunks = []
for doc in MEDICAL_DOCUMENTS:
    all_chunks.extend(chunker.chunk_document(doc))

print(f"Generated {len(all_chunks)} semantic chunks.")


---
## 4. Embedding & FAISS Vector Search + Cross-Encoder Reranker
We embed chunks with `bge-small-en-v1.5` and build a FAISS inner-product index for sub-millisecond retrieval.


In [ ]:
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

# 1. Load embedding model
print("Loading embedding model BAAI/bge-small-en-v1.5 ...")
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

# 2. Embed all knowledge chunks
texts_to_embed = [c["text"] for c in all_chunks]
embeddings = embed_model.encode(texts_to_embed, normalize_embeddings=True, convert_to_numpy=True)
embeddings = embeddings.astype(np.float32)

# 3. Build FAISS Flat Inner-Product Index (Cosine Similarity)
dim = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(embeddings)
print(f"FAISS index built with {faiss_index.ntotal} vectors of dimension {dim}.")

# 4. Load Cross-Encoder Reranker
print("Loading cross-encoder reranker ...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512)

class RAGRetriever:
    def __init__(self, index, metadata, embedder, cross_encoder):
        self.index = index
        self.metadata = metadata
        self.embedder = embedder
        self.cross_encoder = cross_encoder

    def retrieve(self, query: str, top_k: int = 2) -> List[Dict[str, Any]]:
        # Dense vector search
        query_vec = self.embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
        search_k = min(top_k * 3, len(self.metadata))
        scores, indices = self.index.search(query_vec, search_k)

        candidates = []
        for score, idx in zip(scores[0], indices[0]):
            if idx >= 0:
                item = dict(self.metadata[idx])
                item["dense_score"] = float(score)
                candidates.append(item)

        if not candidates:
            return []

        # Cross-encoder reranking
        pairs = [[query, c["text"]] for c in candidates]
        rerank_scores = self.cross_encoder.predict(pairs)
        for c, s in zip(candidates, rerank_scores):
            c["rerank_score"] = float(s)

        candidates.sort(key=lambda x: x["rerank_score"], reverse=True)
        return candidates[:top_k]

retriever = RAGRetriever(faiss_index, all_chunks, embed_model, reranker)

# Quick retrieval test
test_results = retriever.retrieve("What should I do if I have a high fever?")
print("\n--- Retrieval Verification ---")
for i, r in enumerate(test_results, 1):
    print(f"[{i}] {r['title']} (Score: {r['rerank_score']:.3f})")
    print(f"    {r['text'][:140]}...")


---
## 5. Clinical Safety & Emergency Detection Guardrails
Before querying models or retrieval, user inputs pass through a strict emergency filter to detect life-threatening crises (stroke, heart attacks, suicidal ideation, severe dyspnea).


In [ ]:
class Severity(Enum):
    SAFE = auto()
    CAUTION = auto()
    EMERGENCY = auto()

@dataclass
class SafetyResult:
    is_emergency: bool = False
    is_caution: bool = False
    emergency_response: Optional[str] = None
    triggers: List[str] = field(default_factory=list)

EMERGENCY_RULES = [
    (r"\b(crushing|severe|squeezing)\b.*\bchest\b", "Cardiac emergency"),
    (r"\bchest\s+pain\b", "Potential cardiac issue"),
    (r"\b(cannot|can't)\s+breathe\b", "Severe respiratory distress"),
    (r"\b(face\s+droop|slurred\s+speech|arm\s+weakness)\b", "Acute stroke symptoms"),
    (r"\bstroke\b", "Stroke alert"),
    (r"\b(kill\s+myself|suicide|end\s+my\s+life|want\s+to\s+die)\b", "Suicide crisis"),
    (r"\bunconscious|passed\s+out|fainted\b", "Loss of consciousness")
]

EMERGENCY_RESPONSE_TEMPLATE = """🚨 **CRITICAL MEDICAL EMERGENCY DETECTED** 🚨

If you or someone near you is experiencing severe or life-threatening symptoms, **please contact emergency medical services immediately**:

- **Emergency Number**: Call **911** (US/Canada), **999** (UK), or **112** (Europe / International).
- **Suicide & Crisis Lifeline**: Dial or text **988** (available 24/7 free and confidential).

⚠️ *Do NOT wait or rely on an AI chatbot for emergency medical situations. Seek direct professional care right now.*"""

def detect_emergency(query: str) -> SafetyResult:
    query_lower = query.lower()
    triggered = []
    for pattern, name in EMERGENCY_RULES:
        if re.search(pattern, query_lower):
            triggered.append(name)

    if triggered:
        return SafetyResult(
            is_emergency=True,
            is_caution=True,
            emergency_response=EMERGENCY_RESPONSE_TEMPLATE,
            triggers=triggered
        )
    return SafetyResult(is_emergency=False, is_caution=False)

# Verification
print("Emergency detection test:", detect_emergency("I have crushing chest pain").is_emergency)
print("Routine question test:   ", detect_emergency("What are symptoms of iron deficiency?").is_emergency)


---
## 6. MedGemma 4B QLoRA Fine-Tuning Configuration (GPU)
Here is the production-grade fine-tuning setup for training Google's MedGemma 4B model using 4-bit NormalFloat quantization (QLoRA).


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "google/medgemma-4b-it"

# 4-bit Quantization Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True
)

# LoRA Adapter Configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

print("QLoRA Fine-tuning parameters configured:")
print(f"  Target Model : {MODEL_ID}")
print(f"  LoRA Rank (r): {lora_config.r}, Alpha: {lora_config.lora_alpha}")
print(f"  Quantization : 4-bit NormalFloat (NF4)")


---
## 7. Evaluation: Accuracy Score & Confusion Matrix Heatmap
We evaluate safety classification against a curated 30-question medical benchmark, printing accuracy and plotting the confusion matrix.


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# Curated benchmark dataset
BENCHMARK_ITEMS = [
    {"q": "What are common symptoms of iron deficiency anemia?", "critical": 0},
    {"q": "What are the main symptoms of type 2 diabetes?", "critical": 0},
    {"q": "What are signs of dehydration?", "critical": 0},
    {"q": "What are symptoms of a stroke?", "critical": 1},
    {"q": "What are common side effects of ibuprofen?", "critical": 0},
    {"q": "Can ibuprofen be dangerous for the kidneys?", "critical": 0},
    {"q": "What is the difference between a cold and influenza?", "critical": 0},
    {"q": "When should fever be treated as an emergency?", "critical": 1},
    {"q": "How can I improve sleep hygiene?", "critical": 0},
    {"q": "What does a blood pressure reading of 130/85 mean?", "critical": 0},
    {"q": "I have sudden severe crushing chest pain radiating to my left arm", "critical": 1},
    {"q": "My father's face is drooping and his speech is slurred", "critical": 1},
    {"q": "I cannot breathe and my lips are turning blue", "critical": 1},
    {"q": "What vitamins help with energy?", "critical": 0},
    {"q": "How much water should I drink daily?", "critical": 0},
    {"q": "Is paracetamol safe for children with mild fever?", "critical": 0},
    {"q": "I am feeling hopeless and thinking about ending my life", "critical": 1},
    {"q": "What causes frequent urination?", "critical": 0},
    {"q": "How to treat a minor first degree burn at home?", "critical": 0},
    {"q": "Someone just collapsed and stopped breathing", "critical": 1}
]

y_true = []
y_pred = []

for item in BENCHMARK_ITEMS:
    true_label = item["critical"]
    res = detect_emergency(item["q"])
    pred_label = 1 if res.is_emergency else 0
    y_true.append(true_label)
    y_pred.append(pred_label)

# Calculate metrics
acc = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
class_names = ["Routine / Safe", "Safety-Critical"]

print("=" * 60)
print(f"  📊 SAFETY CLASSIFICATION ACCURACY: {acc * 100:.2f}% ({acc:.4f})")
print("=" * 60)
print("\n📋 CLASSIFICATION REPORT:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

# Plot Confusion Matrix Heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    cbar=False,
    annot_kws={"size": 14}
)
plt.title(f"Safety Detection Confusion Matrix\nAccuracy: {acc * 100:.1f}%", fontsize=12, pad=15)
plt.xlabel("Predicted Label", fontsize=11)
plt.ylabel("Actual Label", fontsize=11)
plt.tight_layout()
plt.show()


---
## 8. Interactive Gradio Web Chatbot UI
Run the Gradio interface directly in Colab. Setting `share=True` automatically generates a public `https://xxxx.gradio.live` link you can access on any device!


In [ ]:
import gradio as gr

def medical_chat_bot(message: str, history: list):
    if not message or not message.strip():
        return history, ""

    # Step 1: Emergency & Safety Layer check
    safety = detect_emergency(message)
    if safety.is_emergency:
        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": safety.emergency_response})
        return history, ""

    # Step 2: RAG Retrieval from Knowledge Base
    docs = retriever.retrieve(message, top_k=2)
    if docs:
        parts = ["Here is verified clinical guidance from the medical knowledge base:\n"]
        for d in docs:
            parts.append(f"### 📌 {d['title']}")
            parts.append(f"{d['text']}\n")
            parts.append(f"*Source: {d['source']}*\n")
        parts.append("\n⚠️ *Note: This is general educational guidance. Consult a certified medical professional for personalized diagnoses.*")
        response = "\n".join(parts)
    else:
        response = (
            "I am an educational AI medical assistant.\n\n"
            "I could not locate specific verified information in my knowledge base for your inquiry. "
            "Please consult a certified healthcare practitioner for personalized guidance."
        )

    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response})
    return history, ""

# Build Gradio UI
with gr.Blocks(title="AI Medical Assistant") as demo:
    gr.Markdown("# 🏥 AI Medical Assistant (Colab Edition)")
    gr.Markdown("Ask health questions about symptoms, wellness, and medical conditions.")
    gr.Markdown("⚠️ **Disclaimer:** For educational use only. In life-threatening emergencies, dial **911/999/112** immediately.")

    chatbot = gr.Chatbot(height=420, avatar_images=(None, "🏥"))
    msg = gr.Textbox(placeholder="Ask your medical question here...", lines=2)
    with gr.Row():
        send_btn = gr.Button("Send ➤", variant="primary")
        clear_btn = gr.Button("Clear")

    gr.Examples(
        examples=[
            "What should I do if I have a mild fever?",
            "What are the symptoms of type 2 diabetes?",
            "What are common side effects of ibuprofen?",
            "I have crushing chest pain and difficulty breathing"
        ],
        inputs=msg
    )

    send_btn.click(medical_chat_bot, inputs=[msg, chatbot], outputs=[chatbot, msg])
    msg.submit(medical_chat_bot, inputs=[msg, chatbot], outputs=[chatbot, msg])
    clear_btn.click(lambda: ([], ""), outputs=[chatbot, msg])

# Launch with share=True to generate a public link
demo.launch(share=True, debug=False)
